In [ ]:
!pip install "transformers<4.54.0" trl

In [ ]:
from peft import LoraConfig
peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
# train_dpo.py
from datasets import load_dataset
from trl import DPOConfig, DPOTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl.trainer.utils import SIMPLE_CHAT_TEMPLATE


model = AutoModelForCausalLM.from_pretrained("sft_safety_model")
tokenizer = AutoTokenizer.from_pretrained("sft_safety_model")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.chat_template is None:
    tokenizer.chat_template = SIMPLE_CHAT_TEMPLATE
    
train_dataset = load_dataset("json", data_files="/shared-docker/harmless_test.jsonl", split="train")

training_args = DPOConfig(output_dir="Qwen2-0.5B-DPO")
trainer = DPOTrainer(model=model, args=training_args, processing_class=tokenizer, train_dataset=train_dataset, peft_config=peft_config)
trainer.train()

In [ ]:
import os

print("saving the model and tokenizer...")
trainer.save_model()

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model_name = "sft_safety_model"   # or whatever you used
adapter_model_path = "sft_safety_model_dpo"

# Load base + adapter
model = AutoModelForCausalLM.from_pretrained(base_model_name)
model = PeftModel.from_pretrained(model, adapter_model_path)

# Merge LoRA weights into the base model
model = model.merge_and_unload()

# Save as a normal HF model folder
save_path = "sft_safety_model_dpo_merged"
model.save_pretrained(save_path)

# Also save tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.save_pretrained(save_path)